# Prediction Accuracy Analysis: Polymarket vs. FiveThirtyEight

**DS2500 — Programming with Data | Northeastern University**  
**Team:** Sebastian Brookes, Shiven Mishra

Which source more accurately predicted the actual outcomes of the 2024 U.S. presidential
election? This notebook compares **Polymarket** (prediction market odds) against
**FiveThirtyEight** (polling averages) on state-level winner calls, tracking how their
accuracy evolved over time and culminated on election night.

In [ ]:
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

# add project root to path so we can import from src/
PROJECT_ROOT = str(Path.cwd().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.visualize.accuracy_plots import (
    load_data,
    compute_daily_accuracy,
    build_head_to_head_metrics,
    build_polymarket_trajectory_metrics,
    build_ev_comparison_metrics,
    style_axis,
    annotate_bars,
    CLR_PM,
    CLR_538,
    CLR_TRUMP,
    CLR_HARRIS,
    OVERLAP_STATES,
    SWING_OVERLAP,
    PERIODS,
    OVERLAP_CUTOFF,
)
from src.visualize.event_plots import Theme, apply_style, format_header

THEME = Theme()
apply_style()

# lower DPI for inline display
plt.rcParams["figure.dpi"] = 150

## Methodology

### Data sources

| Source | What it provides | Granularity |
| --- | --- | --- |
| **Polymarket** | Daily Trump win-probability per state | State × day |
| **FiveThirtyEight** | Polling averages (Trump % and Dem %) | State × day |
| **FEC** | Official 2024 election results | State-level ground truth |

### Overlap constraints

Polymarket's state-level markets launched on different dates, so a fair head-to-head
comparison is only possible for the **13 states** where both sources have daily data
through **September 12, 2024** (when the last Polymarket state market opened):
AZ, CA, FL, GA, MI, MN, NC, NH, NV, OH, PA, TX, WI.

Of these, **7 are swing states**: AZ, GA, MI, NC, NV, PA, WI.

### Winner-call methodology

For each state on a given date:
- **Polymarket** → Trump wins if `trump_prob > 0.5`
- **FiveThirtyEight** → Trump wins if `trump_pct > dem_pct`

A call is "correct" if it matches the FEC-certified winner.

### How the pipeline works

The computation functions imported above (from `src/visualize/accuracy_plots.py`) handle
three non-obvious data-wrangling challenges:

1. **Missing-date forward-fill for 538.** Polymarket has one row per state per day, but
   FiveThirtyEight only publishes when new polls arrive — some states have multi-day gaps.
   To compare on any given date, we forward-fill by taking each state's most recent
   available value (`latest_per_state` sorts by `[state, date]` and keeps the last row
   per state up to the as-of date). This avoids dropping states that happen to lack a
   poll on a specific day.

2. **Snapshot → accuracy scoring.** For a given date, we build a "snapshot" of each
   source's predictions, apply the winner-call rule, then inner-join on the FEC results
   by state. Accuracy = number of correct calls / number of states in the intersection.
   The intersection step matters because not every state appears in every source on
   every date.

3. **Electoral-vote aggregation.** For the EV chart, we merge each source's state-level
   winner calls with the FEC's `electoral_votes` column, then sum Trump vs. Harris EVs
   and normalize to percentages of the available pool.

The cell below demonstrates these steps on a concrete example.

In [ ]:
from src.visualize.accuracy_plots import (
    latest_per_state,
    predict_winner_pm,
    predict_winner_538,
    compute_accuracy,
)

# --- demonstrate the pipeline on Sept 12 for a single state (PA) ---

# load raw data (reused in later cells)
pm, p538, fec = load_data()
fec_winners = fec.set_index("state")["winner"]

as_of = "2024-09-12"
state = "PA"

# step 1: Polymarket has an exact row for each state × date
pm_pa = pm[(pm["date"] == pd.Timestamp(as_of)) & (pm["state"] == state)]
print("Step 1 — Polymarket row for PA on Sept 12:")
display(pm_pa[["date", "state", "trump_prob"]])

# step 2: 538 may not have a row on Sept 12. forward-fill grabs the
#          most recent available row per state up to that date.
p538_pa_exact = p538[(p538["date"] == pd.Timestamp(as_of)) & (p538["state"] == state)]
p538_pa_filled = latest_per_state(p538, as_of)
p538_pa_filled = p538_pa_filled[p538_pa_filled["state"] == state]
print(f"\nStep 2 — 538 exact match on Sept 12: {len(p538_pa_exact)} rows")
print(f"         After forward-fill:          {len(p538_pa_filled)} row (from {p538_pa_filled['date'].iloc[0].date()})")
display(p538_pa_filled[["date", "state", "trump_pct", "dem_pct"]])

# step 3: apply winner-call rules
pm_pa_pred = predict_winner_pm(pm_pa)
p538_pa_pred = predict_winner_538(p538_pa_filled)
actual = fec_winners.loc[state]
print(f"\nStep 3 — Winner calls for {state}:")
print(f"  Polymarket:      {pm_pa_pred['predicted_winner'].iloc[0]}  (trump_prob = {pm_pa['trump_prob'].iloc[0]:.2f})")
print(f"  FiveThirtyEight: {p538_pa_pred['predicted_winner'].iloc[0]}  (trump_pct = {p538_pa_filled['trump_pct'].iloc[0]:.1f}, dem_pct = {p538_pa_filled['dem_pct'].iloc[0]:.1f})")
print(f"  Actual winner:   {actual}")

# step 4: score accuracy across all 13 overlap states at once
pm_snap = predict_winner_pm(pm[pm["date"] == pd.Timestamp(as_of)])
p538_snap = predict_winner_538(latest_per_state(p538, as_of))

pm_acc = compute_accuracy(pm_snap, fec_winners, OVERLAP_STATES)
p538_acc = compute_accuracy(p538_snap, fec_winners, OVERLAP_STATES)
print(f"\nStep 4 — Full accuracy across {len(OVERLAP_STATES)} overlap states:")
print(f"  Polymarket:      {pm_acc['correct']}/{pm_acc['n_states']} correct ({pm_acc['pct']:.1f}%)")
print(f"  FiveThirtyEight: {p538_acc['correct']}/{p538_acc['n_states']} correct ({p538_acc['pct']:.1f}%)")

In [ ]:
# pm, p538, fec were loaded in the demo cell above — print summary info

print(f"Polymarket:      {pm.shape[0]:,} rows, {pm.shape[1]} cols")
print(f"FiveThirtyEight: {p538.shape[0]:,} rows, {p538.shape[1]} cols")
print(f"FEC results:     {fec.shape[0]} states")
print()
print(f"Polymarket date range:  {pm['date'].min().date()} to {pm['date'].max().date()}")
print(f"538 date range:         {p538['date'].min().date()} to {p538['date'].max().date()}")
print(f"Overlap states:         {len(OVERLAP_STATES)} ({', '.join(OVERLAP_STATES)})")
print(f"Swing overlap:          {len(SWING_OVERLAP)} ({', '.join(SWING_OVERLAP)})")
print()
print("Polymarket sample:")
display(pm.head())
print("\nFiveThirtyEight sample:")
display(p538.head())
print("\nFEC sample:")
display(fec.head())

In [ ]:
ts = compute_daily_accuracy(pm, p538, fec)
head_to_head = build_head_to_head_metrics(pm, p538, fec)
trajectory = build_polymarket_trajectory_metrics(pm, fec)
ev = build_ev_comparison_metrics(pm, p538, fec)

print(f"Time-series:  {len(ts)} daily observations ({ts['date'].min().date()} to {ts['date'].max().date()})")
print(f"Head-to-head: {len(head_to_head)} groups")
print(f"Trajectory:   {len(trajectory)} snapshots")
print(f"EV compare:   {len(ev)} scenarios")

In [ ]:
# --- Plot 1: Time-Series Crossover (daily accuracy, Mar-Sep 12) ---

fig, ax = plt.subplots(figsize=(14, 7))
fig.subplots_adjust(top=0.82)

ax.plot(
    ts["date"], ts["pm_pct"],
    color=CLR_PM, linewidth=2.5, label="Polymarket", zorder=3,
)
ax.plot(
    ts["date"], ts["p538_pct"],
    color=CLR_538, linewidth=2.5, linestyle="--", dashes=(6, 3),
    label="FiveThirtyEight", zorder=3,
)

# period shading
shade_colors = ["#FFFFFF", "#F4F4F6", "#FFFFFF"]
for i, (label, pstart, pend) in enumerate(PERIODS):
    pstart_ts = pd.Timestamp(pstart)
    pend_ts = pd.Timestamp(pend)
    ax.axvspan(pstart_ts, pend_ts, color=shade_colors[i], alpha=0.6, zorder=0)
    sub = ts[(ts["date"] >= pstart_ts) & (ts["date"] <= pend_ts)]
    if not sub.empty:
        mid_x = pstart_ts + (pend_ts - pstart_ts) / 2
        ax.text(mid_x, 105, label, ha="center", va="bottom",
                fontsize=9, color="#888888")

# Biden dropout marker
biden_date = pd.Timestamp("2024-07-21")
ax.axvline(biden_date, color="#555555", linestyle=":", linewidth=1.5,
           alpha=0.8, zorder=2)
ax.text(
    pd.Timestamp("2024-07-24"), 45, "Biden drops out",
    fontsize=9, color="#555555", fontstyle="italic",
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=2),
    zorder=4,
)

style_axis(ax, y_grid_alpha=0.4)
ax.set_ylim(30, 108)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v)}%"))
ax.set_ylabel("States Predicted Correctly (%)")
ax.set_xlabel("Date (2024)")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.legend(loc="lower left", frameon=False, fontsize=11)

format_header(
    fig,
    "Polymarket Overtook FiveThirtyEight in State-Level Accuracy",
    "13 overlap states, daily winner-call accuracy, Mar–Sep 12 2024.",
)
fig.tight_layout(rect=[0, 0, 1, 0.88])
plt.show()

### Time-series interpretation

The crossover chart reveals a striking reversal. In the stable early months (Mar–May),
FiveThirtyEight's polling averages were nearly flawless — **99.4% daily accuracy** versus
Polymarket's **63.6%**. Polls correctly reflected an electorate that hadn't yet been
disrupted by major events.

Then Biden dropped out on July 21, and everything shifted. In the Aug–Sep 12 period,
Polymarket pulled ahead at **79.2%** versus FiveThirtyEight's **62.2%**. Markets
recalibrated to the new Harris-vs-Trump race within days; poll averages, weighed down by
stale Biden-era surveys, took far longer to catch up.

The lesson: **polls excel in stability, markets excel in transition.**

In [ ]:
# --- Plot 2: Head-to-Head Snapshot (Sept 12) ---

groups = head_to_head["group"].tolist()
pm_vals = head_to_head["pm_pct"].to_numpy()
p538_vals = head_to_head["p538_pct"].to_numpy()

x = np.arange(len(groups))
width = 0.32

fig, ax = plt.subplots(figsize=(6, 5))
fig.subplots_adjust(top=0.82)

bars_pm = ax.bar(x - width / 2, pm_vals, width, label="Polymarket", color=CLR_PM)
bars_538 = ax.bar(x + width / 2, p538_vals, width, label="FiveThirtyEight", color=CLR_538)

annotate_bars(ax, bars_pm, color=CLR_PM, bold=True)
annotate_bars(ax, bars_538, color=CLR_538, bold=True)

ax.set_ylim(0, 105)
ax.set_ylabel("States Predicted Correctly (%)")
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.legend(frameon=False)
style_axis(ax)

format_header(
    fig,
    "State-by-State Prediction Accuracy",
    "Winner calls as of September 12, 2024.",
)
fig.tight_layout(rect=[0, 0, 1, 0.88])
plt.show()

### Head-to-head interpretation

September 12 is the cutoff because it's the date the last Polymarket state market
launched — the first day all 13 overlap states have data from both sources simultaneously.

On that date, Polymarket correctly called **11 of 13 states (84.6%)** versus
FiveThirtyEight's **8 of 13 (61.5%)**.

The gap widens dramatically in the swing states that actually decide the election:
Polymarket got **5 of 7 (71.4%)** correct, while FiveThirtyEight managed only
**2 of 7 (28.6%)**. FiveThirtyEight's polling averages missed NC, NV, PA, MI, and WI —
all states that Trump ultimately won. Polymarket's only misses were MI and WI, where
Trump's margins were razor-thin.

In [ ]:
# --- Plot 3: Polymarket Trajectory (Sept 12 / Oct 6 / Nov 4) ---

labels = trajectory["label"].tolist()
all_vals = trajectory["all_pct"].to_numpy()
swing_vals = trajectory["swing_pct"].to_numpy()

x = np.arange(len(labels))
width = 0.32

fig, ax = plt.subplots(figsize=(7, 5))
fig.subplots_adjust(top=0.82)

bars_all = ax.bar(x - width / 2, all_vals, width,
                  label="Sample States", color=CLR_PM)
bars_sw = ax.bar(x + width / 2, swing_vals, width,
                 label="Swing States", color=CLR_PM, alpha=0.55)

annotate_bars(ax, bars_all, color=CLR_PM, bold=True)
annotate_bars(ax, bars_sw, color=CLR_PM)

final_pct = float(all_vals[-1]) if len(all_vals) else 0.0

ax.set_ylim(0, 115)
ax.set_ylabel("States Predicted Correctly (%)")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(frameon=False)
style_axis(ax)

format_header(
    fig,
    f"Polymarket Reached {final_pct:.0f}% Accuracy by Election Eve",
    "Winner-call accuracy at three snapshots as state coverage expanded.",
)
fig.tight_layout(rect=[0, 0, 1, 0.88])
plt.show()

### Trajectory interpretation

As Polymarket expanded from 13 states to all 50, its accuracy climbed steadily:
**84.6%** on Sept 12 → **94.0%** on Oct 6 → **96.0%** on election eve. By November 4,
markets had correctly called 48 of 50 states.

The persistent misses were **Michigan and Wisconsin** — both states where Trump won by
less than 1 point. These were genuinely toss-up races that neither markets nor polls
reliably resolved before election day.

In [ ]:
# --- Plot 4: Electoral Vote Comparison ---

ev_labels = ev["label"].tolist()
trump_pct = ev["trump_pct"].to_numpy()
harris_pct = ev["harris_pct"].to_numpy()

x = np.arange(len(ev_labels))
width = 0.55

fig, ax = plt.subplots(figsize=(8, 5))
fig.subplots_adjust(top=0.82)

ax.bar(x, trump_pct, width, color=CLR_TRUMP)
ax.bar(x, harris_pct, width, bottom=trump_pct, color=CLR_HARRIS)

# 50% threshold line
ax.axhline(50, color="gray", linewidth=0.8, linestyle=":", zorder=3)
ax.text(len(ev_labels) - 0.5, 51, "50%", fontsize=8, color="gray", va="bottom")

# highlight actual result
actual_idx = len(ev_labels) - 1
ax.axvspan(actual_idx - 0.5, actual_idx + 0.5,
           color="#E8E8E8", alpha=0.5, zorder=0)

# labels inside bars
for i in range(len(ev_labels)):
    ax.text(
        x[i], trump_pct[i] / 2, f"Trump {trump_pct[i]:.1f}%",
        ha="center", va="center", fontsize=9, fontweight="bold", color="white",
    )
    ax.text(
        x[i], trump_pct[i] + harris_pct[i] / 2, f"Harris {harris_pct[i]:.1f}%",
        ha="center", va="center", fontsize=9, fontweight="bold", color="white",
    )

ax.set_ylim(0, 110)
ax.set_ylabel("Share of Available Electoral Votes (%)")
ax.set_xticks(x)
ax.set_xticklabels(ev_labels)
style_axis(ax)

format_header(
    fig,
    "Polymarket Came Closer to the Actual Electoral Vote Split",
    "Normalized EV share implied by each source's state-level winner calls.",
)
fig.tight_layout(rect=[0, 0, 1, 0.88])
plt.show()

### Electoral vote interpretation

This chart translates winner calls into electoral vote implications. On September 12,
Polymarket's state-by-state predictions implied a **Trump-leaning split** — closer to
the actual result where Trump won decisively. FiveThirtyEight's predictions implied a
**Harris-leaning split**, getting the overall direction wrong.

Markets didn't just call more states correctly — they correctly read which *direction*
the electoral map was tilting. Polls, still absorbing the post-Biden-dropout landscape,
pointed the wrong way entirely.

## Conclusions

### Summary of findings

1. **Polymarket was more accurate overall.** On the fairest comparison date (Sept 12),
   markets correctly called 84.6% of states versus 61.5% for polls. In swing states,
   the gap was even wider: 71.4% vs. 28.6%.

2. **The advantage wasn't always markets'.** In the stable early months (Mar–May),
   FiveThirtyEight dominated at 99.4% daily accuracy versus Polymarket's 63.6%. Markets
   only pulled ahead after the Biden dropout reshuffled the race.

3. **Markets correctly read the electoral map direction; polls got it backwards.**
   Polymarket's Sept 12 predictions implied a Trump-leaning electoral vote split, which
   matched the actual outcome. FiveThirtyEight predicted a Harris-leaning split — the
   wrong direction entirely.

4. **Both sources shared blind spots.** Michigan and Wisconsin were missed by both
   Polymarket and FiveThirtyEight at every snapshot. These razor-thin Trump wins
   (< 1 point) were genuinely unpredictable.

### Practical takeaway

Prediction markets and polls have complementary strengths. **Markets excel when the race
is shifting quickly** — they reprice overnight after candidate changes, debates, and
breaking news. **Polls excel in stable periods** where the underlying electorate hasn't
changed and the slow accumulation of survey data converges on the truth. Neither source
alone tells the full story, but understanding *when* each performs best is essential for
anyone building election forecasts.

### Limitations

- **Overlap constraints** — The head-to-head comparison is limited to 13 states through
  Sept 12 because Polymarket's state markets launched at different times. This excludes
  the final two months of the campaign.
- **Binary winner calls** — We reduce rich probability/polling data to binary "Trump or
  Harris" calls. This misses the confidence each source assigned to its prediction.
- **Single election** — All findings are from one election cycle. Prediction markets may
  not maintain this advantage in elections without a major mid-campaign disruption.
- **Apples-to-oranges scales** — Polymarket reports win probabilities while FiveThirtyEight
  reports vote shares. Both get converted to the same binary winner call, but the
  underlying quantities are fundamentally different.

### Further reading

This notebook establishes *what* each source got right and wrong. A companion analysis
([Event Response Analysis](./events.ipynb)) examines *how quickly and intensely* each
source reacted to specific campaign events — measuring reaction speed, z-scored intensity,
and directional accuracy for five major events including the Biden dropout,
the Trump assassination attempt, and the Harris-Trump debate.